# 02 - Coded Chat Exploration

This notebook explores only the predator-coded rows produced by `01_coded_chat_data_loading.ipynb`. It uses no manual labels and performs no model training.

Run notebook 01 first so that `data/processed/coded_stage_messages.csv` and `data/processed/coded_chat_summary.csv` exist.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

WORKSPACE = Path.cwd()
PROCESSED_DIR = WORKSPACE / "data" / "processed"
REPORT_DIR = WORKSPACE / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

messages_path = PROCESSED_DIR / "coded_stage_messages.csv"
summary_path = PROCESSED_DIR / "coded_chat_summary.csv"

if not messages_path.exists() or not summary_path.exists():
    raise FileNotFoundError("Run 01_coded_chat_data_loading.ipynb before this notebook.")

messages = pd.read_csv(messages_path, dtype={"category": "string"})
chat_summary = pd.read_csv(summary_path)


## Inventory and class balance

These statistics identify class scarcity and recording gaps before Stage 1 modeling. Accuracy alone is not a useful summary because 900 messages may be rare.


In [ ]:
VALID_CATEGORIES = {"200", "600", "900"}

assert messages["category"].isin(VALID_CATEGORIES).all()
assert not messages.duplicated(["chat_id", "msg_idx"]).any()
assert set(messages["chat_id"]).issubset(set(chat_summary["chat_id"]))

overall = pd.DataFrame({
    "metric": [
        "coded_chat_count",
        "coded_message_count",
        "chats_reaching_600",
        "chats_reaching_900",
        "timestamped_messages",
        "timestamp_coverage_pct",
    ],
    "value": [
        messages["chat_id"].nunique(),
        len(messages),
        messages.loc[messages["category"] == "600", "chat_id"].nunique(),
        messages.loc[messages["category"] == "900", "chat_id"].nunique(),
        messages["datetime_text"].notna().sum(),
        round(100 * messages["datetime_text"].notna().mean(), 2),
    ],
})

class_balance = (
    messages.groupby("category", observed=True)
    .size()
    .rename("message_count")
    .reset_index()
)
class_balance["message_share_pct"] = (
    100 * class_balance["message_count"] / class_balance["message_count"].sum()
).round(2)

display(overall)
display(class_balance.sort_values("category"))


## Per-chat stage reachability and ordering

Retained predator messages preserve their source indices. Ordering checks are descriptive; later modeling may smooth isolated reversals, but source codes are never overwritten.


In [ ]:
def analyze_chat(group: pd.DataFrame) -> pd.Series:
    ordered = group.sort_values("msg_idx")
    codes = ordered["category"].astype(int).tolist()

    onset_200 = min([int(i) for i, code in enumerate(codes, start=1) if code == 200], default=np.nan)
    onset_600 = min([int(i) for i, code in enumerate(codes, start=1) if code == 600], default=np.nan)
    onset_900 = min([int(i) for i, code in enumerate(codes, start=1) if code == 900], default=np.nan)

    known_onsets = [value for value in (onset_200, onset_600, onset_900) if not pd.isna(value)]
    transitions = list(zip(codes[:-1], codes[1:]))
    reversals = sum(next_code < previous_code for previous_code, next_code in transitions)
    skips = sum(abs(next_code - previous_code) > 400 for previous_code, next_code in transitions)

    return pd.Series({
        "message_count": len(ordered),
        "first_200_position": onset_200,
        "first_600_position": onset_600,
        "first_900_position": onset_900,
        "reached_600": int(600 in codes),
        "reached_900": int(900 in codes),
        "strictly_monotonic": int(all(a <= b for a, b in transitions)),
        "transition_reversal_count": reversals,
        "cross_stage_skip_count": skips,
        "stage_sequence": " -> ".join(map(str, codes)),
    })

chat_diagnostics = (
    messages.groupby("chat_id", sort=False, group_keys=False)
        .apply(analyze_chat)
    .reset_index()
)

chat_diagnostics["chat_id"] = chat_diagnostics["chat_id"].astype(str)
display(chat_diagnostics.describe(include="all").T)
display(chat_diagnostics[["chat_id", "message_count", "reached_600", "reached_900", "strictly_monotonic", "transition_reversal_count"]])


## Saved reports

The saved tables support the write-up without exposing assumptions or replacing source codes. Stage sequences remain compact and descriptive.


In [ ]:
class_balance.to_csv(REPORT_DIR / "coded_stage_class_balance.csv", index=False)
chat_diagnostics.to_csv(REPORT_DIR / "coded_chat_stage_diagnostics.csv", index=False)
chat_summary.to_csv(REPORT_DIR / "coded_chat_inventory.csv", index=False)
overall.to_csv(REPORT_DIR / "coded_dataset_overview.csv", index=False)

print("Saved coded-chat exploration reports to:", REPORT_DIR)
print("Class counts:")
print(class_balance.to_string(index=False))
print("Chats containing reversals:", int((chat_diagnostics["transition_reversal_count"] > 0).sum()))


## Interpretation limits

* This sample includes only 56 available coded files unless the folder is updated.
* All coded chats are predatory, so Stage 1 output cannot measure benign-versus-predatory separation.
* Small class counts require repeated grouped cross-validation and uncertainty intervals.
* Sequence reversals describe existing codes and should not be "fixed" manually.
* Message timestamps are used only where the XML actually contains them.
